# 🧬 VARIANT-GNN Google Colab Training
**Genetik Varyant Patojenite Tahmini - Cloud GPU Eğitimi**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msgxr/VARIANT-GNN/blob/main/VARIANT_GNN_Colab_Training.ipynb)

---
⚠️ **Runtime Settings**: Hardware accelerator: **GPU (T4)** seçili olduğunu kontrol edin!

💰 **Predicted Cost**: ~$3-5 for 20k variant training (Pro+ recommended)

## 🔧 1. SYSTEM SETUP

In [ ]:
# Hardware check
import sys
print(f"🐍 Python: {sys.version}")

import torch
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🖥️  CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️  WARNING: GPU not available! Change runtime to GPU.")

## 📦 2. INSTALL PACKAGES

In [ ]:
# Install PyTorch with CUDA support
!pip install torch==2.2.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install PyTorch Geometric
!pip install torch-geometric --quiet
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.2.0+cu121.html --quiet

print("✅ PyTorch ecosystem installed")

In [ ]:
# Install ML and explainability packages
!pip install xgboost==2.0.3 --quiet
!pip install scikit-learn==1.4.0 --quiet
!pip install shap==0.45.0 --quiet
!pip install lime==0.2.0.1 --quiet
!pip install imbalanced-learn --quiet

print("✅ ML packages installed")

In [ ]:
# Install utilities and visualization
!pip install pandas numpy matplotlib seaborn plotly --quiet
!pip install pydantic omegaconf tqdm --quiet
!pip install fpdf2 networkx --quiet

print("✅ All packages installed successfully!")

## 📂 3. PROJECT & DATA SETUP

In [ ]:
# Clone VARIANT-GNN repository
!git clone https://github.com/msgxr/VARIANT-GNN.git
%cd VARIANT-GNN

# Check project structure
!ls -la
print("\n📊 Data files:")
!ls -lh data/

In [ ]:
# Optional: Mount Google Drive for backup
from google.colab import drive
import os

drive.mount('/content/drive')

# Create backup directory
backup_dir = "/content/drive/MyDrive/VARIANT_GNN_BACKUP"
!mkdir -p "{backup_dir}"

print(f"✅ Google Drive mounted. Backup dir: {backup_dir}")

## ⚙️ 4. COLAB-OPTIMIZED CONFIGURATION

In [ ]:
# Create optimized config for Colab
import yaml

colab_config = {
    # Training settings
    "training": {
        "batch_size": 64,                # T4 GPU optimized
        "max_epochs": 50,               # Session timeout consideration
        "early_stopping_patience": 10,
        "learning_rate": 0.001,
        "weight_decay": 1e-4
    },
    
    # Model architecture (memory efficient)
    "gnn": {
        "hidden_dim": 128,
        "num_layers": 3,
        "dropout": 0.3,
        "use_skip_connections": True
    },
    
    "dnn": {
        "hidden_layers": [256, 128, 64],
        "dropout": 0.4,
        "batch_norm": True
    },
    
    # Ensemble (speed focused)
    "ensemble": {
        "weights": [0.5, 0.3, 0.2],      # XGBoost weight increased
        "optimize_weights": False
    },
    
    # Preprocessing
    "preprocessing": {
        "scaling_method": "robust",
        "smote_enabled": True,
        "use_autoencoder": False,        # Memory saving
        "missing_value_strategy": "median"
    },
    
    # Graph construction
    "graph": {
        "similarity_metric": "cosine",
        "k_neighbors": 8,              # Reduced complexity
        "similarity_threshold": 0.3
    },
    
    # System optimization
    "system": {
        "dataloader_workers": 2,
        "pin_memory": True,
        "mixed_precision": True,
        "gradient_accumulation_steps": 2
    }
}

# Save config
with open('configs/colab_config.yaml', 'w') as f:
    yaml.dump(colab_config, f, default_flow_style=False)

print("✅ Colab-optimized configuration created!")
print(f"📊 Config preview: Batch size: {colab_config['training']['batch_size']}")

## 📈 5. RESOURCE MONITORING

In [ ]:
# Resource monitoring functions
import psutil
import torch
from datetime import datetime

def monitor_resources():
    """Monitor system resources"""
    # Memory
    memory = psutil.virtual_memory()
    print(f"💾 RAM: {memory.used/1024**3:.1f}/{memory.total/1024**3:.1f} GB ({memory.percent:.1f}%)")
    
    # GPU
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory
        gpu_used = torch.cuda.memory_allocated(0)
        print(f"🖥️  GPU: {gpu_used/1024**3:.1f}/{gpu_memory/1024**3:.1f} GB ({gpu_used/gpu_memory*100:.1f}%)")
    
    # Disk
    disk = psutil.disk_usage('/')
    print(f"💿 Disk: {disk.used/1024**3:.1f}/{disk.total/1024**3:.1f} GB ({disk.percent:.1f}%)")
    print(f"⏰ Time: {datetime.now().strftime('%H:%M:%S')}")
    print("-" * 50)

def cleanup_memory():
    """Clean up GPU and system memory"""
    import gc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("🧹 Memory cleaned up")

# Initial resource check
print("📊 INITIAL RESOURCE STATUS:")
monitor_resources()

## 🎯 6. MODEL TRAINING

In [ ]:
# Quick test training (small dataset)
print("🧪 QUICK TEST TRAINING (5 epochs)")
print("="*50)

!python main.py \
    --mode train \
    --data_file data/train_general.csv \
    --config_path configs/colab_config.yaml \
    --max_epochs 5 \
    --batch_size 32 \
    --verbose

print("\n✅ Quick test completed!")
monitor_resources()

In [ ]:
# Create checkpoint backup function
import shutil
from datetime import datetime

def create_checkpoint(description=""):
    """Create a checkpoint backup"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_name = f"checkpoint_{timestamp}_{description}"
    checkpoint_path = f"/content/drive/MyDrive/VARIANT_GNN_BACKUP/{checkpoint_name}"
    
    # Create directories
    !mkdir -p "{checkpoint_path}"
    
    # Backup important files
    if os.path.exists('models/'):
        !cp -r models/ "{checkpoint_path}/"
    if os.path.exists('reports/'):
        !cp -r reports/ "{checkpoint_path}/"
    !cp -r configs/ "{checkpoint_path}/"
    
    print(f"💾 Checkpoint created: {checkpoint_name}")
    return checkpoint_path

# Create initial checkpoint
checkpoint_path = create_checkpoint("after_test")
cleanup_memory()

In [ ]:
# Full dataset training
import time

print("🚀 FULL DATASET TRAINING STARTING...")
print("="*50)
print(f"📊 Expected time: 4-6 hours")
print(f"💰 Expected cost: ~$3-5")
print("⚠️  Keep this tab open to avoid session timeout")
print()

start_time = time.time()

# Monitor resources before training
monitor_resources()

# Main training command
!python main.py \
    --mode train \
    --data_file data/train_variants.csv \
    --config_path configs/colab_config.yaml \
    --max_epochs 50 \
    --panel General \
    --save_best_model \
    --early_stopping \
    --verbose

end_time = time.time()
training_duration = end_time - start_time

print("\n" + "="*50)
print("🎉 TRAINING COMPLETED!")
print(f"⏱️  Total time: {training_duration/3600:.2f} hours")
print(f"💰 Estimated cost: ${training_duration/3600 * 1.5:.2f}")
print("="*50)

# Final resource check
monitor_resources()

# Create final checkpoint
final_checkpoint = create_checkpoint("full_training_complete")
cleanup_memory()

## 📊 7. RESULTS EVALUATION

In [ ]:
# Display training results
import json
import pandas as pd
import matplotlib.pyplot as plt

print("🎯 TRAINING RESULTS")
print("="*50)

# Load and display CV results
if os.path.exists('reports/cv_report.json'):
    with open('reports/cv_report.json', 'r') as f:
        cv_results = json.load(f)
    
    print("📈 Cross-Validation Results:")
    for metric, value in cv_results.items():
        if isinstance(value, dict):
            print(f"\n{metric.upper()}:")
            for k, v in value.items():
                print(f"  {k}: {v:.4f}")
        elif isinstance(value, (int, float)):
            print(f"{metric}: {value:.4f}")
else:
    print("❌ CV report not found")

# Check model files
print("\n📁 Generated Model Files:")
!ls -lh models/

print("\n📋 Report Files:")
!ls -lh reports/

In [ ]:
# Test prediction on blind test set
print("🔮 BLIND TEST PREDICTION")
print("="*50)

# Make predictions
!python main.py \
    --mode predict \
    --test_file data/test_variants_blind.csv \
    --output_dir reports/ \
    --load_models

# Display prediction results
if os.path.exists('reports/predictions.csv'):
    pred_df = pd.read_csv('reports/predictions.csv')
    
    print(f"\n📊 PREDICTION SUMMARY:")
    print(f"Total variants: {len(pred_df)}")
    print(f"Predicted Pathogenic: {(pred_df['Prediction'] == 1).sum()}")
    print(f"Predicted Benign: {(pred_df['Prediction'] == 0).sum()}")
    print(f"Average Confidence: {pred_df['Confidence'].mean():.3f}")
    
    # High confidence predictions
    high_conf = pred_df[pred_df['Confidence'] > 0.8]
    print(f"High confidence (>0.8): {len(high_conf)} variants")
    
    print("\n🎯 First 10 Predictions:")
    print(pred_df.head(10))
    
    # Plot confidence distribution
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    pred_df['Confidence'].hist(bins=20, alpha=0.7, color='skyblue')
    plt.title('Confidence Distribution')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    
    plt.subplot(1, 2, 2)
    pred_counts = pred_df['Prediction'].value_counts()
    plt.pie(pred_counts.values, labels=['Benign', 'Pathogenic'], autopct='%1.1f%%')
    plt.title('Prediction Distribution')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ Prediction file not generated")

## 💾 8. DOWNLOAD RESULTS

In [ ]:
# Create downloadable zip file
import zipfile
from google.colab import files

zip_name = f"VARIANT_GNN_Results_{datetime.now().strftime('%Y%m%d_%H%M')}.zip"

with zipfile.ZipFile(zip_name, 'w') as zipf:
    # Add model files
    for root, dirs, files_list in os.walk('models/'):
        for file in files_list:
            zipf.write(os.path.join(root, file))
    
    # Add reports
    for root, dirs, files_list in os.walk('reports/'):
        for file in files_list:
            zipf.write(os.path.join(root, file))
    
    # Add config
    zipf.write('configs/colab_config.yaml')

print(f"📦 Results packaged: {zip_name}")
print(f"📊 File size: {os.path.getsize(zip_name)/1024/1024:.1f} MB")

# Download file
files.download(zip_name)
print("⬇️ Download started!")

## ✅ 9. FINAL STATUS & CLEANUP

In [ ]:
# Final resource monitor
print("📊 FINAL SYSTEM STATUS:")
monitor_resources()

# Session summary
print("\n🎉 SESSION SUMMARY:")
print("="*50)
print("✅ Project cloned successfully")
print("✅ Dependencies installed")
print("✅ Model training completed")
print("✅ Predictions generated")
print("✅ Results backed up to Drive")
print("✅ Results downloaded")

# Cleanup
cleanup_memory()
print("\n🧹 Memory cleaned up")

print("\n🎯 NEXT STEPS:")
print("1. Review downloaded results")
print("2. Check model performance metrics")
print("3. Use models for further predictions")
print("4. Access backups from Google Drive if needed")

print("\n🚀 Training completed successfully! 🎊")